# Drive Daily Stats — DDL

**Purpose:**  
Create the target table for clean, typed daily Backblaze drive observations in the Curated/Silver layer.

**Source:** `backblaze_data.ing_bronze.drive_observations`  
**Target:** `backblaze_data.sot_silver.drive_daily_stats`

## Responsibilities

- Define a stable target schema based on the unified Bronze source schema.
- Convert `date` to `DATE`, `capacity_bytes` to `BIGINT`, and `failure` to `INT`.
- Type SMART normalized attributes as `BIGINT`; retain SMART raw attributes as `STRING` to avoid overflow or loss of source representation.
- Preserve `_period`, `_source_file_path`, and `_ingested_at` for lineage.
- Create the table only if it does not already exist; do not load or overwrite data.

## Grain

One observation per `(_period, date, serial_number)`.

Grain enforcement and data-quality routing belong in the functional ETL notebook.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DateType, LongType, IntegerType, TimestampType
)

BRONZE_TABLE = "backblaze_data.ing_bronze.drive_observations"
TARGET_TABLE = "backblaze_data.sot_silver.drive_daily_stats"

if not spark.catalog.tableExists(BRONZE_TABLE):
    raise ValueError(f"Required source table does not exist: {BRONZE_TABLE}")

bronze_schema = spark.table(BRONZE_TABLE).schema
bronze_columns = [field.name for field in bronze_schema.fields]

required_columns = {
    "date", "serial_number", "model", "capacity_bytes", "failure",
    "_source_file_path", "_ingested_at", "_period"
}
missing = required_columns - set(bronze_columns)
if missing:
    raise ValueError(f"Bronze table is missing required columns: {sorted(missing)}")

print(f"Source table: {BRONZE_TABLE}")
print(f"Target table: {TARGET_TABLE}")
print(f"Bronze columns discovered: {len(bronze_columns)}")

In [0]:
def target_type(column_name):
    if column_name == "date":
        return DateType()
    if column_name == "capacity_bytes":
        return LongType()
    if column_name == "failure":
        return IntegerType()
    if column_name.endswith("_normalized") and column_name.startswith("smart_"):
        return LongType()
    if column_name == "_ingested_at":
        return TimestampType()
    return StringType()

target_fields = [
    StructField(field.name, target_type(field.name), nullable=True)
    for field in bronze_schema.fields
]
target_schema = StructType(target_fields)

# print("Curated target schema:")
# target_schema.simpleString()

In [0]:
# Creating an empty Delta table with the target schema.
if not spark.catalog.tableExists(TARGET_TABLE):
    (
        spark.createDataFrame([], target_schema)
        .write
        .format("delta")
        .saveAsTable(TARGET_TABLE)
    )

    print(f"Created empty table: {TARGET_TABLE}")
else:
    print(f"Table already exists; left unchanged: {TARGET_TABLE}")

display(spark.sql(f"DESCRIBE TABLE {TARGET_TABLE}"))


In [0]:
target_df = spark.table(TARGET_TABLE)
target_columns = set(target_df.columns)

if target_columns != set(bronze_columns):
    raise ValueError(
        "Target columns do not match the expected Bronze-derived contract. "
        f"Missing={sorted(set(bronze_columns) - target_columns)}, "
        f"Unexpected={sorted(target_columns - set(bronze_columns))}"
    )

expected_types = {
    "date": "date",
    "capacity_bytes": "bigint",
    "failure": "int",
    "_ingested_at": "timestamp",
    "_period": "string",
    "_source_file_path": "string",
    "serial_number": "string",
}
actual_types = {field.name: field.dataType.simpleString() for field in target_df.schema.fields}
for column_name, expected_type in expected_types.items():
    if actual_types[column_name] != expected_type:
        raise ValueError(
            f"Unexpected type for {column_name}: {actual_types[column_name]} "
            f"(expected {expected_type})"
        )

print(f"DDL validation passed. Columns: {len(target_df.columns)}")
print(f"Current target row count: {target_df.count():,}")
